## Evaluator-Optimizer Workflow
In this workflow, one LLM call generates a response while another provides evaluation and feedback in a loop.

### When to use this workflow
This workflow is particularly effective when we have:

- Clear evaluation criteria
- Value from iterative refinement

The two signs of good fit are:

- LLM responses can be demonstrably improved when feedback is provided
- The LLM can provide meaningful feedback itself

In [1]:
#!pip install anthropic

### Get API Key

Buy $5 credits. 
https://console.anthropic.com/dashboard

In [2]:
# Set environment varaible. Needed by util.py

import os

ANTHROPIC_API_KEY = "xxx"

In [52]:
from util import llm_call, extract_xml

def generate(prompt: str, task: str, context: str = "") -> tuple[str, str]:
    """Generate and improve a solution based on feedback."""
    full_prompt = f"{prompt}\n{context}\nTask: {task}" if context else f"{prompt}\nTask: {task}"
    response = llm_call(full_prompt)
    
    # Extract the sections
    thoughts = extract_xml(response, "thoughts")
    
    # Adding support for additional tags in the result
    unit_tests = extract_xml(response, "UNIT_TESTS")  # Extract unit tests
    code = extract_xml(response, "CODE")  # Extract code block
    gherkin = extract_xml(response, "GHERKIN")  # Extract Gherkin format if applicable
    step_impl = extract_xml(response, "STEP_IMPL")
    
    # Print the result with new tags
    print("\n=== GENERATION START ===")
    print(f"Thoughts:\n{thoughts}\n")
    print(f"Generated Code:\n{code}")
    if unit_tests:
        print(f"Unit Tests:\n{unit_tests}")
    if gherkin:
        print(f"Gherkin Scenario:\n{gherkin}")
    if step_impl:
        print(f"Step impl:\n{step_impl}")
    print("=== GENERATION END ===\n")
    
    return thoughts, unit_tests, code, gherkin, step_impl


def evaluate(prompt: str, content: str, task: str) -> tuple[str, str]:
    """Evaluate if a solution meets requirements based on code, unit tests, and Gherkin scenarios."""
    full_prompt = f"{prompt}\nOriginal task: {task}\nContent to evaluate: {content}"
    response = llm_call(full_prompt)
    
    # Extract evaluation and feedback for all components
    evaluation = extract_xml(response, "evaluation")
    feedback = extract_xml(response, "feedback")
    
    # Print the evaluation result
    print("=== EVALUATION START ===")
    print(f"Status: {evaluation}")
    print(f"Feedback: {feedback}")
    print("=== EVALUATION END ===\n")
    
    return evaluation, feedback


def loop(task: str, evaluator_prompt: str, generator_prompt: str) -> tuple[str, list[dict]]:
    """Keep generating and evaluating until requirements are met."""
    memory = [None, (None, None, None, None)]
    chain_of_thought = []
    
    thoughts, unit_tests, code, gherkin, step_impl = generate(generator_prompt, task)
    memory = [thoughts,(unit_tests, code, gherkin,step_impl)]
    chain_of_thought = ({
        "thoughts": thoughts,
        "unit_tests": unit_tests,
        "code": code,
        "gherkin": gherkin,
        "step_impl":step_impl
    })
    
    while True:
        evaluation, feedback = evaluate(evaluator_prompt, memory, task)
        if evaluation == "PASS":
            return memory, chain_of_thought, unit_tests, code, gherkin, step_impl
            
        context = "\n".join([
            "Previous attempts:",
            *[f"- {m}" for m in memory],
            f"\nFeedback: {feedback}"
        ])
        
        thoughts, unit_tests, code, gherkin,step_impl = generate(generator_prompt, task, context)
        memory = [thoughts,(unit_tests, code, gherkin)]
        chain_of_thought = ({
            "thoughts": thoughts,
            "unit_tests": unit_tests,
            "code": code,
            "gherkin": gherkin,
            "step_impl":step_impl
        })

        break

### Example Use Case: Iterative coding loop



In [53]:
evaluator_prompt = """
Evaluate the following components based on the task:
1. <CODE>: Is the implementation of the code correct and does it meet the task requirements? 
2. <UNIT_TESTS>: Are the unit tests sufficient to test the implementation? Do they cover edge cases?
3. <GHERKIN>: Does the Gherkin scenario correctly represent the expected behavior? Is it clear and comprehensive?
4. <STEP_IMPL>: Does the impl steps fully wir the code and gherkin together?

You should evaluate each component individually:
- For <CODE>, consider correctness, efficiency, and best practices.
- For <UNIT_TESTS>, check the coverage and clarity of the tests.
- For <GHERKIN>, check if the scenario matches the described behavior.
- For <STEP_IMPL>, chck that gherkin and code are wired together.

Output your evaluation in the following format:

<evaluation>PASS, NEEDS_IMPROVEMENT, or FAIL</evaluation>
<feedback>
Provide feedback for each component (code, unit tests, Gherkin, and step_impl) with specific suggestions for improvement.
</feedback>

Do not attempt to solve the task. Only evaluate based on the components.
"""


generator_prompt = """
Your goal is to complete the task based on <user input>. If there are feedback 
from your previous generations, you should reflect on them to improve your solution

Output your answer concisely in the following format: 

<thoughts>
[Your understanding of the task and feedback and how you plan to improve]
</thoughts>

<CODE>
[Put exe code here]
</CODE>

<UNIT_TESTS>
[Put unit tests here. They must cover CODE completely]
</UNIT_TESTS>

<GHERKIN>
[Put gherkin here]
</GHERKIN>

<STEP_IMPL>
[Wiring for gherkin and code here]
</STEP_IMPL>
"""

task = """
<user input>
Implement a Stack with:
1. push(x)
2. pop()
3. getMin()
All operations should be O(1).
</user input>
"""

# task = """
# <user input>
# Implement a python program to retrieve stock data.
# 1. Use yfinance to acqure data for QQQ
# 2. Plot the results with plotly
# All operations should be O(1).
# </user input>
# """

loop(task, evaluator_prompt, generator_prompt)



=== GENERATION START ===
Thoughts:

To implement a Stack with O(1) operations including getMin(), I'll use two stacks:
- Main stack for regular push/pop
- Min stack to track minimum values
Each time we push, we'll also push to min stack if value <= current min
This ensures getMin() is always O(1)


Generated Code:

```python
class MinStack:
    def __init__(self):
        self.stack = []
        self.min_stack = []
    
    def push(self, x: int) -> None:
        self.stack.append(x)
        if not self.min_stack or x <= self.min_stack[-1]:
            self.min_stack.append(x)
            
    def pop(self) -> int:
        if not self.stack:
            raise IndexError("Stack is empty")
        if self.stack[-1] == self.min_stack[-1]:
            self.min_stack.pop()
        return self.stack.pop()
    
    def getMin(self) -> int:
        if not self.min_stack:
            raise IndexError("Stack is empty")
        return self.min_stack[-1]
```

Unit Tests:

```python
import pytest


(["\nTo implement a Stack with O(1) operations including getMin(), I'll use two stacks:\n- Main stack for regular push/pop\n- Min stack to track minimum values\nEach time we push, we'll also push to min stack if value <= current min\nThis ensures getMin() is always O(1)\n",
  ('\n```python\nimport pytest\n\ndef test_min_stack():\n    stack = MinStack()\n    \n    # Test empty stack\n    with pytest.raises(IndexError):\n        stack.pop()\n    with pytest.raises(IndexError):\n        stack.getMin()\n        \n    # Test push and min\n    stack.push(3)\n    assert stack.getMin() == 3\n    stack.push(2)\n    assert stack.getMin() == 2\n    stack.push(5)\n    assert stack.getMin() == 2\n    \n    # Test pop\n    assert stack.pop() == 5\n    assert stack.getMin() == 2\n    assert stack.pop() == 2\n    assert stack.getMin() == 3\n    assert stack.pop() == 3\n    \n    # Test empty again\n    with pytest.raises(IndexError):\n        stack.getMin()\n```\n',
   '\n```python\nclass MinStack:\n 